# 01 — Data CollectionThis notebook collects live UK job postings from the **JSearch API** (via RapidAPI) for 5 data-related roles:- Data Analyst- Data Engineer- Data Scientist- AI Engineer- ML EngineerThe API aggregates listings from LinkedIn, Indeed, Glassdoor and other job boards. We pull jobs posted in the **last 30 days**, restricted to **Great Britain (gb)**.**Requires:** a `.env` file in the project root with `RAPIDAPI_KEY=your_key_here` (see `.env.example`).

In [1]:
import osimport timeimport requestsimport pandas as pdfrom pathlib import Pathfrom dotenv import load_dotenvprint("UK JOB MARKET DATA COLLECTION — STARTED")

UK JOB MARKET DATA COLLECTION — STARTED

In [2]:
BASE_DIR = Path.cwd()load_dotenv(BASE_DIR / ".env")RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")print("API key loaded:", RAPIDAPI_KEY is not None)

API key loaded: True

In [3]:
url = "https://jsearch.p.rapidapi.com/search"headers = {    "X-RapidAPI-Key": RAPIDAPI_KEY,    "X-RapidAPI-Host": "jsearch.p.rapidapi.com"}search_queries = [    "data analyst in United Kingdom",    "data engineer in United Kingdom",    "data scientist in United Kingdom",    "ai engineer in United Kingdom",    "ml engineer in United Kingdom"]

In [4]:
all_jobs = []for query in search_queries:    print(f"Collecting: {query}")    params = {        "query": query,        "page": "1",        "num_pages": "3",        "country": "gb",        "date_posted": "month"    }    response = requests.get(url, headers=headers, params=params, timeout=30)    print("Status:", response.status_code)    if response.status_code != 200:        print(response.text[:300])        continue    data = response.json()    jobs = data.get("data", [])    print("Jobs found:", len(jobs))    for job in jobs:        all_jobs.append({            "job_id":          job.get("job_id"),            "job_title":       job.get("job_title"),            "company":         job.get("employer_name"),            "location":        job.get("job_location"),            "country":         job.get("job_country"),            "employment_type": job.get("job_employment_type"),            "job_description": job.get("job_description"),            "date_posted":     job.get("job_posted_at_datetime_utc"),            "job_url":         job.get("job_apply_link"),            "publisher":       job.get("job_publisher"),            "is_remote":       job.get("job_is_remote"),            "salary_min":      job.get("job_min_salary"),            "salary_max":      job.get("job_max_salary"),            "salary_period":   job.get("job_salary_period"),            "search_query":    query,            "role_category":   query.replace(" in United Kingdom", "").title(),            "api_source":      "JSearch"        })    time.sleep(2)

Collecting: data analyst in United KingdomStatus: 200Jobs found: 88Collecting: data engineer in United KingdomStatus: 200Jobs found: 315Collecting: data scientist in United KingdomStatus: 200Jobs found: 135Collecting: ai engineer in United KingdomStatus: 200Jobs found: 202Collecting: ml engineer in United KingdomStatus: 200Jobs found: 39

In [5]:
df = pd.DataFrame(all_jobs)before = len(df)df.drop_duplicates(subset=["job_id"], inplace=True)after = len(df)print(f"Rows collected: {before}")print(f"Rows after de-duplication: {after}")

Rows collected: 779\nRows after de-duplication: 729

In [6]:
output_path = BASE_DIR / "data" / "raw" / "uk_jobs_raw.csv"output_path.parent.mkdir(parents=True, exist_ok=True)df.to_csv(output_path, index=False, encoding="utf-8")print("RAW JOB DATA SAVED SUCCESSFULLY")print("Total jobs:", len(df))print("Saved to:", output_path)df.head()

RAW JOB DATA SAVED SUCCESSFULLY\nTotal jobs: 729\nSaved to: data/raw/uk_jobs_raw.csv

**Output of this notebook:** `data/raw/uk_jobs_raw.csv` — 729 raw job postings across 5 roles, ready for cleaning in `02_data_cleaning.ipynb`.